# Benchmarking MDS reconstruction

U-Chrom's bulk MDS module (`uchrom.recon.bulk.mds`) exposes two solvers
(SMACOF and Adam) and runs on CPU, CUDA, or Apple MPS.  When you set up
a new analysis you usually want to know:

1. **Which solver should I use?**  (SMACOF vs Adam)
2. **Does the GPU actually help for my data size?**
3. **How does resolution trade off against cost and reconstruction fidelity?**

This notebook is a reproducible benchmark template.  We use GM12878
in-situ Hi-C (Rao et al., 2014) and chromosome 22 so the whole notebook
finishes in under a minute.  The same pattern applies to any
chromosome or dataset.


## Background: evolution from miniMDS

U-Chrom's bulk MDS solver borrows the overall recipe from
[miniMDS (Rieber & Mahony, 2017)](https://github.com/seqcode/miniMDS):
convert contact frequencies to distances via `d = c^(-1/α)`, smooth
with a distance-decay prior, then iteratively minimise MDS stress.
The U-Chrom implementation is a PyTorch-native rewrite with the
following departures:

| Aspect | miniMDS | U-Chrom |
|--------|---------|---------|
| Input | BED text (manual export) | `.hic` / `.mcool` / `.cool` directly |
| Initialisation | 4 random inits × 300 iters | CMDS eigendecomposition × 1 run (up to 1000 iters, early stop on convergence) |
| Device | CPU only (sklearn) | PyTorch on CUDA / MPS / CPU with auto-detection |
| Distance-decay prior | Python double `for` loop | Vectorised `scatter_add_` |
| Zero-contact bins | `assert` crash on all-zero rows | Auto-filtered before MDS; bins returned in `nonzero_mask` |
| Stress | Treats missing pairs as `d = 0` | Observed-pairs-only (weighted mask) |
| Whole-genome | Manual per-chromosome script | `inter_mds()` one-liner |
| Output | Custom TSV with NaN markers | `.chromdata.zarr` (ChromData) |

Both solvers exposed in `run_mds(method=...)` share this infrastructure:

- `method='smacof'` — Guttman transform, no autograd, the direct
  miniMDS lineage path.  Fastest for typical cases.
- `method='adam'` — PyTorch autograd + Adam optimiser.  Useful if you
  need to modify the stress function or add priors.

Typical speedup on GM12878 chr22 @ 100 kb (single-thread CPU):
miniMDS ~0.8 s vs U-Chrom SMACOF ~0.1 s; on Apple MPS / CUDA the
advantage grows to 20–40× at finer resolutions.  Structures remain
equivalent (pairwise-distance Pearson r > 0.99, input-output
correlation within 0.001).


## Locate the GM12878 `.hic` file

The benchmark needs the GM12878 in-situ combined Hi-C file
(`GSE63525_GM12878_insitu_primary+replicate_combined_30.hic`,
~40 GB).  Because of its size this file is **not** auto-downloaded;
drop it into `example-data/` (or point `UCHROM_GM12878_HIC` at it).

Download sources:

- [4DN DCIC — 4DNFI1UEG1HD](https://data.4dnucleome.org/files-processed/4DNFI1UEG1HD/)
- [GEO GSE63525](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE63525)

Also required: `pip install hic-straw` (reads `.hic` files).


In [ ]:
import os
from pathlib import Path

def find_repo_root(start: Path = Path.cwd()) -> Path:
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists() or (p / 'example-data').exists():
            return p
    raise FileNotFoundError('Could not locate repo root')

repo = find_repo_root()
default_path = repo / 'example-data' / 'GSE63525_GM12878_insitu_primary+replicate_combined_30.hic'
HIC_PATH = Path(os.environ.get('UCHROM_GM12878_HIC', str(default_path)))

if not HIC_PATH.exists():
    raise FileNotFoundError(
        f'GM12878 .hic file not found at {HIC_PATH}.\n'
        'Download from 4DN DCIC (4DNFI1UEG1HD) or GEO GSE63525, place it at\n'
        f'  {default_path}\n'
        'or set UCHROM_GM12878_HIC to its location.'
    )
print('Using:', HIC_PATH)


## Load chr22 contact matrix at 100 kb

`uchrom.io.load_hic` wraps `hicstraw` and returns a `(bins_df, matrix)`
tuple.  We'll use chr22 because it's small (~51 Mb → 514 bins at 100 kb)
and good enough to show the scaling trend.


In [ ]:
import numpy as np
import torch
from uchrom.io import load_hic

CHROM, RES = 'chr22', 100_000
bins_df, contact_mat = load_hic(str(HIC_PATH), RES, CHROM)

n = contact_mat.shape[0]
sparsity = 1.0 - np.count_nonzero(contact_mat) / (n * n)
print(f'{CHROM} @ {RES//1000}kb: {n} bins, sparsity {sparsity:.1%}')


## Compare the two MDS solvers

`run_mds` exposes both SMACOF (default, fast) and Adam (gradient
descent with autograd).  Both are called via the same interface; only
the `method` argument differs.


In [ ]:
import time
from uchrom.recon.bulk.mds import run_mds

def time_mds(method, device='cpu', n_iter=1000):
    t0 = time.time()
    coords, mask = run_mds(
        torch.from_numpy(contact_mat.astype(np.float64)),
        alpha=4.0, weight=0.05,
        method=method, device=device, n_iter=n_iter,
        verbose=False,
    )
    return coords, mask, time.time() - t0

coords_smacof, mask, t_smacof = time_mds('smacof', device='cpu')
coords_adam,   _,    t_adam   = time_mds('adam',   device='cpu')

print(f'SMACOF (CPU): {t_smacof:.2f}s, output shape {coords_smacof.shape}')
print(f'Adam   (CPU): {t_adam:.2f}s, output shape {coords_adam.shape}')


## Compare CPU vs GPU (SMACOF)

On Apple silicon `device='auto'` picks MPS (Metal); on an NVIDIA box it
picks CUDA.  For chr22 the matrix is small enough that GPU startup
overhead dominates — the advantage kicks in at finer resolutions (see
the sweep below).


In [ ]:
coords_cpu, _, t_cpu = time_mds('smacof', device='cpu')
coords_gpu, _, t_gpu = time_mds('smacof', device='auto')
print(f'SMACOF CPU : {t_cpu:.2f}s')
print(f'SMACOF auto: {t_gpu:.2f}s  (speedup {t_cpu/t_gpu:.2f}×)')


## Three orthogonal evaluation metrics

No single number captures reconstruction quality.  We compute three:

1. **Agreement (Pearson r on pairwise distances)** — after Procrustes
   alignment, how similar are two reconstructions of the same data?
2. **Input-output correlation** — per method, Spearman correlation
   between observed contact frequency and `1 / d_3D`.  Measures how
   faithfully the 3D structure preserves Hi-C signal (independent of
   the other method).
3. **Distance-fitting RMSD** — how well do the coordinates satisfy
   each solver's own target distance matrix (i.e. the MDS objective)?


In [ ]:
from scipy.spatial.distance import pdist
from scipy.stats import pearsonr, spearmanr
from uchrom.recon.bulk.mds.transforms import procrustes_alignment, apply_transform


def agreement_pearson(a, b):
    """Pairwise distance Pearson r after Procrustes alignment."""
    R, t, s = procrustes_alignment(a, b)
    a_aligned = apply_transform(a, R, t, s)
    return pearsonr(pdist(a_aligned), pdist(b))[0]


def io_correlation(coords, contact):
    """Spearman between contact frequency and 1/d_3D on observed pairs."""
    from scipy.spatial.distance import squareform
    d = squareform(pdist(coords))
    mask = contact > 0
    np.fill_diagonal(mask, False)
    c = contact[mask]
    inv_d = 1.0 / np.clip(d[mask], 1e-12, None)
    return spearmanr(c, inv_d)[0]


def fit_rmsd(coords, target_dist):
    """Normalized RMSD between 3D distances and target on observed pairs."""
    from scipy.spatial.distance import squareform
    d = squareform(pdist(coords))
    mask = (target_dist > 0)
    np.fill_diagonal(mask, False)
    if not mask.any():
        return float('nan')
    scale = target_dist[mask].mean() / max(d[mask].mean(), 1e-12)
    d *= scale
    rg = float(np.sqrt(np.mean(np.sum((coords - coords.mean(0)) ** 2, axis=1))))
    return float(np.sqrt(((d[mask] - target_dist[mask]) ** 2).mean()) / max(rg, 1e-12))


# Build the target distance matrix the way the SMACOF solver saw it
from uchrom.recon.bulk.mds.core import (
    apply_distance_decay_prior, contact_to_distance, normalize_distances,
)

contact_tensor = torch.from_numpy(contact_mat.astype(np.float64))
contact_tensor = contact_tensor[mask][:, mask]  # use the filtered bins
contact_adj = apply_distance_decay_prior(contact_tensor, 0.05)
target_dist = normalize_distances(contact_to_distance(contact_adj, 4.0)).numpy()
contact_sub = contact_tensor.numpy()

print(f'Agreement (SMACOF vs Adam, Pearson r)   : {agreement_pearson(coords_smacof, coords_adam):.4f}')
print(f'IO correlation  — SMACOF                : {io_correlation(coords_smacof, contact_sub):.4f}')
print(f'IO correlation  — Adam                  : {io_correlation(coords_adam,   contact_sub):.4f}')
print(f'Distance-fit RMSD — SMACOF              : {fit_rmsd(coords_smacof, target_dist):.4f}')
print(f'Distance-fit RMSD — Adam                : {fit_rmsd(coords_adam,   target_dist):.4f}')


## Resolution sweep

At coarse resolutions miniMDS-style solvers finish quickly and GPU
overhead dominates.  At fine resolutions GPU wins by a large margin.
Let's quantify that on chr22 across four resolutions.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

records = []
for r in [100_000, 50_000, 25_000, 10_000]:
    bins_r, mat_r = load_hic(str(HIC_PATH), r, CHROM)
    n_r = mat_r.shape[0]

    coords_cpu_r, _, t_cpu_r = time_mds('smacof', device='cpu')
    tensor_r = torch.from_numpy(mat_r.astype(np.float64))
    t0 = time.time()
    coords_gpu_r, mask_r = run_mds(tensor_r, alpha=4.0, weight=0.05,
                                   method='smacof', device='auto',
                                   n_iter=1000, verbose=False)
    t_gpu_r = time.time() - t0

    sub = mat_r[mask_r][:, mask_r]
    io_r = io_correlation(coords_gpu_r, sub)
    records.append({'resolution_kb': r // 1000, 'n_bins': n_r,
                    't_cpu': t_cpu_r, 't_gpu': t_gpu_r,
                    'io_corr': io_r})

df = pd.DataFrame(records).sort_values('resolution_kb', ascending=False)
df


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(df['resolution_kb'], df['t_cpu'], 'o-', label='CPU (1 thread)', color='#E74C3C')
ax1.plot(df['resolution_kb'], df['t_gpu'], 'o-', label='GPU / MPS',     color='#F39C12')
ax1.set_xlabel('Resolution (kb)'); ax1.set_ylabel('Time (s)')
ax1.set_xscale('log'); ax1.set_yscale('log'); ax1.invert_xaxis()
ax1.grid(True, which='both', alpha=0.3); ax1.legend()
ax1.set_title(f'{CHROM} — speed vs resolution')

ax2.plot(df['resolution_kb'], df['io_corr'], 'o-', color='#27AE60')
ax2.set_xlabel('Resolution (kb)'); ax2.set_ylabel('Input-output Spearman r')
ax2.set_xscale('log'); ax2.invert_xaxis()
ax2.set_ylim(0, 1.02); ax2.grid(True, which='both', alpha=0.3)
ax2.set_title(f'{CHROM} — quality vs resolution')

fig.tight_layout()


## Standard output format: `.chromdata.zarr`

U-Chrom's reconstruction output is a ChromData store (`.chromdata.zarr`;
older `.h5cd` files are still read) that every reconstructor (NucDyn, GEM, bulk MDS) and every import
path (FOF-CT, Hi-C .pairs) writes in the same schema.  That means you
can feed any reconstruction to the same downstream tools (loop / TAD
/ compartment callers, browser, plotting helpers) without format
adapters.

Each store carries:

- **coords** — per-bin 3D coordinates
- **spots** — `chrom`, `start`, `end`, `trace_id` (+ optional `cell_id`)
- **format header** — `uchrom_format_version` attr for forward-compatible upgrades
- **optional** — cell metadata, epigenomic tracks, analysis results (loops/TADs/compartments)

`save_particles()` auto-detects the format from the filename
extension (`.chromdata.zarr` / `.cdz` → ChromData, `.csv` → flat CSV).


In [ ]:
from uchrom.io import save_particles, read_particles
from uchrom import ChromData

out_dir = repo / 'tutorials' / '_out_benchmark'
out_dir.mkdir(parents=True, exist_ok=True)

def to_df(coords, bin_mask):
    kept = bins_df[bin_mask].reset_index(drop=True)
    return pd.DataFrame({
        'chrom': kept['chrom'].values,
        'start': kept['start'].values,
        'end':   kept['end'].values,
        'x': coords[:, 0], 'y': coords[:, 1], 'z': coords[:, 2],
    })

save_particles(to_df(coords_smacof, mask), out_dir / 'smacof.chromdata.zarr')
save_particles(to_df(coords_adam,   mask), out_dir / 'adam.chromdata.zarr')

print('Wrote:')
print(f'  {out_dir}/smacof.chromdata.zarr')
print(f'  {out_dir}/adam.chromdata.zarr')


### Reading the output back

Two paths depending on what you need next:

- **`read_particles(path)`** — returns a flat `pandas.DataFrame`
  (chrom/start/end/x/y/z).  Simplest for quick analysis or export to
  another tool.
- **`ChromData.read(path)`** — returns the full ChromData container
  with cell/trace hierarchy, metadata accessors, and
  `compute_distances()` for on-demand pairwise distance matrices.


In [ ]:
# Quick flat-DataFrame access
df_smacof = read_particles(str(out_dir / 'smacof.chromdata.zarr'))
print(df_smacof.head(3))
print('shape:', df_smacof.shape)


In [ ]:
# Full ChromData container
cd = ChromData.read(str(out_dir / 'smacof.chromdata.zarr'))
print(cd)
print('n_traces :', len(cd.traces))
print('n_spots  :', cd.n_spots)


## Visualize side-by-side in the U-Chrom Browser

The browser is a PyQt5 + PyVista GUI that renders any number of
ChromData files as linked 3D viewports.  Launch it from a terminal:

```bash
python -m uchrom.browser tutorials/_out_benchmark/smacof.chromdata.zarr \
                         tutorials/_out_benchmark/adam.chromdata.zarr
```

Inside the browser:

- Each file gets its own viewport (linked camera by default — drag in
  one viewport rotates all of them in sync).
- Toggle **Canvas → Link canvases** to control this behaviour.
- Use the right-side **Properties** panel to change style (line / tube
  / points), width, opacity, colour mode (rainbow / single / per-trace),
  and layer label font/position.  In link mode those settings apply to
  every viewport at once.
- Click a viewport to switch the active canvas (its parameters load
  into the right panel).
- Drag a layer into the **Layers** list to manage visibility or delete
  it.

Tip: open more than two files to see all reconstructions at once; the
browser handles 4–8 viewports comfortably.

The screenshot below shows the two reconstructions for this tutorial
(SMACOF left, Adam right) with cameras synced:

![U-Chrom Browser — SMACOF vs Adam reconstructions of GM12878 chr22 @ 100 kb](_out_benchmark/browser_demo.png)


In [ ]:
# Exact command printed again for easy copy-paste
print('python -m uchrom.browser',
      out_dir / 'smacof.chromdata.zarr', out_dir / 'adam.chromdata.zarr')


## Takeaways and next steps

- **SMACOF is the default** because it converges in fewer iterations
  without autograd overhead.  Adam remains useful when you need a
  custom stress function.
- **GPU wins at finer resolutions.**  For coarse maps (≥ 250 kb) CPU
  is often faster because MPS/CUDA startup overhead dominates.
- **All three metrics agree on the rough conclusion** that SMACOF and
  Adam produce structurally equivalent reconstructions (Pearson r > 0.99)
  — they differ mostly in how quickly they get there.
- For very high resolutions (5 kb and below) dense MDS on a whole
  chromosome is memory-bound.  Use
  `uchrom.recon.bulk.mds.partitioned_mds` (TAD-wise divide and
  conquer) or the whole-genome workflow in `uchrom.recon.bulk.mds.inter`.
- External comparisons against `miniMDS` follow the same pattern — add
  a third call to whatever solver you want and run it through the same
  three metrics.
